In [ ]:
import sys
from pathlib import Path
carpeta_pipeline = next(ruta for ruta in [Path.cwd(), Path.cwd() / 'pipeline_ingesta', Path.cwd() / 'notebook/unidad_IV/pipeline_ingesta'] if (ruta / 'auditoria_pipeline.py').is_file())
sys.path.insert(0, str(carpeta_pipeline.resolve()))
from auditoria_pipeline import iniciar_etapa

auditoria_etapa = iniciar_etapa('04_staging', 'invoice_lines')

# 04 - Staging de InvoiceLines

Ejecutar despues del profiling y de `03_cuarentena_reproceso_invoice_line.ipynb`.

**Entrada:** `raw.invoice_lines`, excluyendo `quarantine.invoice_lines` y errores criticos de `raw.validacion_calidad` por `id_raw` y `lote_id`.

**Salida:** `staging.invoice_lines`. `id_linea` conserva el identificador RAW del flujo original.

Cada carga vuelve a comprobar las exclusiones y retira registros previamente cargados que ahora esten rechazados. Una cuarentena historica sigue bloqueando el registro, incluso si cambia su estado.

In [1]:
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, SERVIDOR, BD_ORIGEN, BD_DESTINO

engine_origen, engine_destino = obtener_engines()
print(f'Conexiones establecidas:')
print(f'  Servidor: {SERVIDOR}')
print(f'  Origen: {BD_ORIGEN}')
print(f'  Destino: {BD_DESTINO}')

Conexion OK (origen): WideWorldImporters
Conexion OK (destino): WideWorldImportersDW2
Conexiones establecidas:
  Servidor: localhost\SQLEXPRESS02
  Origen: WideWorldImporters
  Destino: WideWorldImportersDW2


## 1. Leer datos crudos de raw.invoice_lines

In [ ]:
import pandas as pd
import json
from sqlalchemy import text

query_raw = """
SELECT r.id_raw, r.lote_id, r.payload
FROM raw.invoice_lines r
WHERE NOT EXISTS (
    SELECT 1 FROM quarantine.invoice_lines q
    WHERE q.id_raw = r.id_raw AND q.lote_id = r.lote_id
)
AND NOT EXISTS (
    SELECT 1 FROM raw.validacion_calidad v
    WHERE v.id_raw = r.id_raw AND v.lote_id = r.lote_id
      AND v.es_critico = 1
)
ORDER BY r.id_raw
"""

df_raw = pd.read_sql_query(query_raw, engine_destino)
print(f'Registros de invoice_lines habilitados para staging: {len(df_raw)}')

Total de registros en raw.invoice_lines: 100
Primer payload (primeros 200 caracteres):
  {"ExtendedPrice": "2645.0", "InvoiceID": "1", "InvoiceLineID": "1", "LastEditedWhen": "2013-01-01T12:00:00", "Quantity": "10", "StockItemID": "67", "TaxAmount": "345.0", "TaxRate": "15.0", "UnitPrice"...


## 2. Descomprimir payloads JSON

In [15]:
# Descomprimir JSONs
datos_lista = []
errores_descompresion = []

for idx, row in df_raw.iterrows():
    try:
        payload = json.loads(row['payload'])
        payload['id_raw'] = row['id_raw']
        datos_lista.append(payload)
    except Exception as e:
        errores_descompresion.append({
            'id_raw': row['id_raw'],
            'error': str(e)
        })

df_descomprimido = pd.DataFrame(datos_lista)
print(f"Registros descomprimidos exitosamente: {len(df_descomprimido)}")
print(f"Errores de descompresión: {len(errores_descompresion)}")

if len(df_descomprimido) > 0:
    print(f"\nEstructura de datos descomprimidos:")
    print(f"  Columnas: {list(df_descomprimido.columns)}")
    print(f"\nMuestra de datos:")
    display(df_descomprimido.head(3))

Registros descomprimidos exitosamente: 100
Errores de descompresión: 0

Estructura de datos descomprimidos:
  Columnas: ['ExtendedPrice', 'InvoiceID', 'InvoiceLineID', 'LastEditedWhen', 'Quantity', 'StockItemID', 'TaxAmount', 'TaxRate', 'UnitPrice', 'id_raw']

Muestra de datos:


,ExtendedPrice,InvoiceID,InvoiceLineID,LastEditedWhen,Quantity,StockItemID,TaxAmount,TaxRate,UnitPrice,id_raw
0,2645.0,1,1,2013-01-01T12:00:00,10,67,345.0,15.0,-230.0,1
1,134.55,2,2,2013-01-01T12:00:00,9,50,17.55,15.0,13.0,2
2,331.2,2,3,2013-01-01T12:00:00,9,10,43.2,15.0,32.0,3


## 3. Normalizar y transformar datos

In [ ]:
columnas_necesarias = ['id_raw', 'InvoiceID', 'StockItemID', 'Quantity', 'ExtendedPrice', 'TaxAmount']
df_aceptados = df_descomprimido.reindex(columns=columnas_necesarias).copy()

for campo in ['id_raw', 'InvoiceID', 'StockItemID', 'Quantity', 'ExtendedPrice', 'TaxAmount']:
    df_aceptados[campo] = pd.to_numeric(df_aceptados[campo], errors='raise')

df_aceptados['neto'] = df_aceptados['ExtendedPrice'] - df_aceptados['TaxAmount']
print(f'Registros para staging: {len(df_aceptados)}')
display(df_aceptados.head())

Registros para STAGING: 100

Datos normalizados:


,id_raw,InvoiceID,StockItemID,Quantity
0,1,1,67,10
1,2,2,50,9
2,3,2,10,9
3,4,3,114,3
4,5,4,206,96


## 4. Preparar para guardar en staging

In [17]:
# Seleccionar y renombrar columnas
columnas_mapping = {
    'id_raw': 'id_linea',
    'InvoiceID': 'id_factura',
    'CustomerID': 'id_cliente',
    'StockItemID': 'id_producto',
    'Quantity': 'cantidad',
    'neto': 'neto',
    'TaxAmount': 'impuesto',
    'ExtendedPrice': 'total'
}

# Tomar solo las columnas que existen
cols_existentes = [c for c in columnas_mapping.keys() if c in df_aceptados.columns]
staging_final = df_aceptados[cols_existentes].copy()
staging_final = staging_final.rename(columns={k: v for k, v in columnas_mapping.items() if k in cols_existentes})

print(f"Estructura final para STAGING:")
print(staging_final.dtypes)
print(f"\nMuestra:")
display(staging_final.head())

Estructura final para STAGING:
id_linea         int64
id_factura         str
id_producto        str
cantidad           str
neto           float64
impuesto       float64
total          float64
dtype: object

Muestra:


,id_linea,id_factura,id_producto,cantidad,neto,impuesto,total
0,1,1,67,10,2300.0,345.00,2645.00
1,2,2,50,9,117.0,17.55,134.55
2,3,2,10,9,288.0,43.20,331.20
3,4,3,114,3,90.0,13.50,103.50
4,5,4,206,96,259.2,38.88,298.08


## 5. Crear tabla staging.invoice_lines

In [ ]:
create_table_sql = """
IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'staging')
    EXEC('CREATE SCHEMA staging');

IF OBJECT_ID('staging.invoice_lines', 'U') IS NULL
BEGIN
    CREATE TABLE staging.invoice_lines (
        id_linea INT PRIMARY KEY,
        id_factura INT,
        id_cliente INT,
        id_producto INT,
        cantidad INT,
        neto DECIMAL(18, 2),
        impuesto DECIMAL(18, 2),
        total DECIMAL(18, 2),
        timestamp_carga DATETIME DEFAULT GETDATE()
    );
END;
"""

with engine_destino.begin() as conn:
    conn.execute(text(create_table_sql))
print('[OK] Tabla staging.invoice_lines verificada/creada')

Error al crear tabla: Not an executable object: "\nIF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'staging')\n    EXEC('CREATE SCHEMA staging');\n\nIF NOT EXISTS (SELECT * FROM INFORMATION_SCHEMA.TABLES \n              WHERE TABLE_SCHEMA = 'staging' AND TABLE_NAME = 'invoice_lines')\nBEGIN\n    CREATE TABLE staging.invoice_lines (\n        id_linea INT PRIMARY KEY,\n        id_factura INT,\n        id_cliente INT,\n        id_producto INT,\n        cantidad INT,\n        neto DECIMAL(18, 2),\n        impuesto DECIMAL(18, 2),\n        total DECIMAL(18, 2),\n        timestamp_carga DATETIME DEFAULT GETDATE()\n    )\nEND\n"


## 6. Guardar datos en SQL

In [ ]:
sql_retirar_rechazados = """
DELETE FROM staging.invoice_lines
WHERE id_linea IN (
    SELECT r.id_raw
    FROM raw.invoice_lines r
    WHERE EXISTS (
        SELECT 1 FROM quarantine.invoice_lines q
        WHERE q.id_raw = r.id_raw AND q.lote_id = r.lote_id
    ) OR EXISTS (
        SELECT 1 FROM raw.validacion_calidad v
        WHERE v.id_raw = r.id_raw AND v.lote_id = r.lote_id
          AND v.es_critico = 1
    )
)
"""

with engine_destino.begin() as conn:
    retirados = conn.execute(text(sql_retirar_rechazados)).rowcount
    habilitados = pd.read_sql_query(text(query_raw), conn)['id_raw']
    ids_existentes = pd.read_sql_query(
        'SELECT id_linea FROM staging.invoice_lines', conn
    )['id_linea']
    filas_nuevas = staging_final.drop_duplicates(subset=['id_linea'])
    filas_nuevas = filas_nuevas.loc[
        filas_nuevas['id_linea'].isin(habilitados)
        & ~filas_nuevas['id_linea'].isin(ids_existentes)
    ]
    if not filas_nuevas.empty:
        filas_nuevas.to_sql(
            'invoice_lines', con=conn, schema='staging',
            if_exists='append', index=False
        )

print(f'[OK] {retirados} rechazados retirados; {len(filas_nuevas)} registros nuevos en staging')

[OK] 0 registros nuevos guardados; 100 ya existían.


## 7. Validar datos guardados

In [ ]:
query_control = """
SELECT COUNT(*) AS rechazados_en_staging
FROM staging.invoice_lines s
INNER JOIN raw.invoice_lines r ON r.id_raw = s.id_linea
WHERE EXISTS (
    SELECT 1 FROM quarantine.invoice_lines q
    WHERE q.id_raw = r.id_raw AND q.lote_id = r.lote_id
) OR EXISTS (
    SELECT 1 FROM raw.validacion_calidad v
    WHERE v.id_raw = r.id_raw AND v.lote_id = r.lote_id
      AND v.es_critico = 1
)
"""
with engine_destino.connect() as conn:
    rechazados = conn.execute(text(query_control)).scalar_one()
if rechazados != 0:
    raise RuntimeError(f'Hay {rechazados} registros rechazados dentro de staging.invoice_lines')
print('[OK] Cero registros de cuarentena o con errores criticos en staging.invoice_lines')

query_validar = """
SELECT COUNT(*) AS total_registros,
       COUNT(DISTINCT id_factura) AS facturas_unicas,
       SUM(CAST(cantidad AS BIGINT)) AS cantidad_total,
       SUM(neto) AS neto_total,
       SUM(impuesto) AS impuesto_total,
       SUM(total) AS total_general
FROM staging.invoice_lines
"""
display(pd.read_sql_query(query_validar, engine_destino))
display(pd.read_sql_query(
    'SELECT TOP 10 * FROM staging.invoice_lines ORDER BY id_linea', engine_destino
))

In [ ]:
auditoria_etapa.finalizar(globals())